# QuantAlpha AI — Step 7: XGBoost Classifier (Proper Time-Split Validation)

**Goal:** predict UP/DOWN over the next 15 trading days using a real ML model, trained on
everything collected so far (technical + fundamental features), validated the same rigorous way
as Step 6 — train on old data, test ONLY on unseen future data, no shortcuts.

**Important, upfront:** this WILL show you a real accuracy number — but based on everything we've
tested (Steps 3B, 4C, 6), realistic outcomes for this kind of task are in the 52-58% range, not
80-90%. If this model shows something dramatically higher, that's a signal to suspect a bug
(likely a feature leaking future information), not a reason to celebrate — we'll check for that
explicitly in Section 6.


## 1. Connect to Drive + install

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

!pip install xgboost scikit-learn --quiet
print("Ready.")


Mounted at /content/drive
Ready.


In [2]:
import pandas as pd
import numpy as np
import glob
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')


## 2. Build the training dataset from your 10-year data (Step 6)
For every stock, every date, we compute the features available AT that date, and the label
(did the stock go up over the NEXT 15 days). This is standard supervised learning setup for
time series — one row per (stock, date).


In [3]:
fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
fscore_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['piotroski_f_score']))
roce_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['roce']))

technical_10yr_files = glob.glob('data_10yr/technical/*.parquet')
symbols_10yr = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_10yr_files)
print(f"Building training data from {len(symbols_10yr)} stocks")

HOLDING_DAYS = 15
all_rows = []

for sym in symbols_10yr:
    df = pd.read_parquet(f"data_10yr/technical/{sym}.parquet")
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df.reset_index(drop=True)
    if len(df) < 300:
        continue

    fscore = fscore_lookup.get(sym, np.nan)
    roce = roce_lookup.get(sym, np.nan)

    for i in range(200, len(df) - HOLDING_DAYS):
        row = {
            'symbol': sym,
            'date': df.loc[i, 'Date'],
            'rsi_14': df.loc[i, 'RSI_14'],
            'adx_14': df.loc[i, 'ADX_14'],
            'atr_14': df.loc[i, 'ATR_14'],
            'close_to_sma50': (df.loc[i, 'Close'] - df.loc[i, 'SMA_50']) / df.loc[i, 'SMA_50'] if pd.notna(df.loc[i, 'SMA_50']) and df.loc[i, 'SMA_50'] != 0 else np.nan,
            'close_to_sma200': (df.loc[i, 'Close'] - df.loc[i, 'SMA_200']) / df.loc[i, 'SMA_200'] if pd.notna(df.loc[i, 'SMA_200']) and df.loc[i, 'SMA_200'] != 0 else np.nan,
            'piotroski_f_score': fscore,
            'roce': roce,
        }
        exit_price = df.loc[i + HOLDING_DAYS, 'Close']
        entry_price = df.loc[i, 'Close']
        fwd_return = (exit_price - entry_price) / entry_price
        row['target'] = 1 if fwd_return > 0 else 0
        row['fwd_return'] = fwd_return
        all_rows.append(row)

data = pd.DataFrame(all_rows)
data = data.dropna(subset=['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200'])
print(f"Training dataset built: {len(data)} rows")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")
print(f"Target balance: {data['target'].value_counts(normalize=True).round(3).to_dict()}")


Building training data from 188 stocks
Training dataset built: 397238 rows
Date range: 2017-05-03 00:00:00 to 2026-06-17 00:00:00
Target balance: {1: 0.55, 0: 0.45}


## 3. Time-based train/test split — the non-negotiable rule
Train on everything before 2024. Test ONLY on 2024 onward — data the model has never seen,
mimicking how it would actually be used going forward.


In [4]:
FEATURES = ['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200', 'piotroski_f_score', 'roce']

train = data[data['date'] < '2024-01-01'].copy()
test = data[data['date'] >= '2024-01-01'].copy()

print(f"Train set: {len(train)} rows ({train['date'].min()} to {train['date'].max()})")
print(f"Test set:  {len(test)} rows ({test['date'].min()} to {test['date'].max()})")

# Fill remaining NaNs (mainly piotroski/roce for stocks missing fundamental data) with median
for col in ['piotroski_f_score', 'roce']:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)


Train set: 283160 rows (2017-05-03 00:00:00 to 2023-12-29 00:00:00)
Test set:  114078 rows (2024-01-01 00:00:00 to 2026-06-17 00:00:00)


## 4. Train XGBoost
Modest depth/complexity on purpose — a shallow model is less likely to overfit noisy financial
data, and gives us more trustworthy feature importances.


In [5]:
model = xgb.XGBClassifier(
    max_depth=4,
    n_estimators=200,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(train[FEATURES], train['target'])
print("Model trained.")


Model trained.


## 5. Evaluate on the UNSEEN test set (2024 onward)

In [6]:
preds = model.predict(test[FEATURES])
probs = model.predict_proba(test[FEATURES])[:, 1]

accuracy = accuracy_score(test['target'], preds)
auc = roc_auc_score(test['target'], probs)

print(f"=== XGBoost Results on UNSEEN 2024-2026 data ===\n")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc:.3f}  (0.5 = random guessing, 1.0 = perfect)")
print(f"\n{classification_report(test['target'], preds, target_names=['DOWN', 'UP'])}")

print("\nFeature importance:")
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importance)


=== XGBoost Results on UNSEEN 2024-2026 data ===

Accuracy: 52.85%
AUC-ROC: 0.509  (0.5 = random guessing, 1.0 = perfect)

              precision    recall  f1-score   support

        DOWN       0.48      0.09      0.15     53378
          UP       0.53      0.92      0.67     60700

    accuracy                           0.53    114078
   macro avg       0.51      0.50      0.41    114078
weighted avg       0.51      0.53      0.43    114078


Feature importance:
close_to_sma200      0.154159
atr_14               0.153988
adx_14               0.151084
roce                 0.144430
close_to_sma50       0.143784
rsi_14               0.136499
piotroski_f_score    0.116056
dtype: float32


## 6. Sanity check — is this accuracy suspiciously high?
If accuracy is above ~65%, that's a red flag for a bug (likely feature leakage), not genuine
skill, given everything we've tested so far. This cell checks for the most common leakage causes.


In [7]:
print("=== Leakage sanity checks ===\n")

# Check 1: does test period overlap with train period at all?
overlap = set(train['date']).intersection(set(test['date']))
print(f"1. Date overlap between train/test: {len(overlap)} dates (should be 0)")

# Check 2: are any features suspiciously perfectly correlated with the target?
for feat in FEATURES:
    corr = train[[feat, 'target']].corr().iloc[0, 1]
    flag = " <-- SUSPICIOUS, investigate" if abs(corr) > 0.3 else ""
    print(f"2. Correlation({feat}, target): {corr:.3f}{flag}")

# Check 3: compare to a trivial baseline (always predict majority class)
majority_class_accuracy = max(test['target'].mean(), 1 - test['target'].mean())
print(f"\n3. Majority-class baseline accuracy: {majority_class_accuracy*100:.2f}%")
print(f"   Model accuracy:                   {accuracy*100:.2f}%")
print(f"   Real edge over trivial baseline:  {(accuracy - majority_class_accuracy)*100:+.2f} points")


=== Leakage sanity checks ===

1. Date overlap between train/test: 0 dates (should be 0)
2. Correlation(rsi_14, target): 0.010
2. Correlation(adx_14, target): 0.033
2. Correlation(atr_14, target): -0.008
2. Correlation(close_to_sma50, target): -0.009
2. Correlation(close_to_sma200, target): 0.009
2. Correlation(piotroski_f_score, target): 0.002
2. Correlation(roce, target): -0.022

3. Majority-class baseline accuracy: 53.21%
   Model accuracy:                   52.85%
   Real edge over trivial baseline:  -0.36 points


## 7. How to read this honestly

- **The real number that matters is Section 6's "edge over trivial baseline"** — not raw accuracy.
  If the market went up 55% of days in the test period, a model that always predicts "UP" gets
  55% accuracy for free, with zero intelligence. Only the edge ABOVE that baseline reflects real
  predictive skill.
- **AUC-ROC is often a more honest metric than accuracy** for this kind of task — 0.52-0.58 is a
  realistic, respectable range; anything near 0.5 means no real skill; anything above ~0.70
  should trigger suspicion of leakage, given everything we've tested
- **Feature importance** tells you what the model is actually relying on — if `piotroski_f_score`
  or `roce` rank highly, that's consistent with Step 3C's finding that fundamental quality is your
  most genuine signal; if a technical feature dominates, treat that cautiously given Steps 3B/6
  showed no standalone technical edge


## 8. Summary
This is now your most rigorous, honest predictive model:
- Real ML (XGBoost), not just rule-based thresholds
- Proper time-split validation (train on 2016-2023, test on unseen 2024-2026)
- Explicit leakage checks built in, not just a number reported blindly
- Compared against a trivial baseline, not celebrated in isolation

**Whatever number came out, that's your real, defensible accuracy claim** — the one you can state
confidently in an interview because you can also explain exactly how it was validated and what
its limitations are.

**Next: GitHub + README** — time to write up the whole journey, including this model, as your
portfolio narrative.
